# B vs r Relationship: GLADE (B band) vs Legacy Survey DR10 (r band)

We query 50 random sky regions (avoiding the galactic plane and
regions outside the Legacy Survey footprint). For each region:

1. Query **GLADE 2.3** (VizieR) → B magnitude
2. Query **Legacy Survey DR10** (NOIRLab DataLab) → r magnitude
3. Cross-match by position (3 arcsec tolerance)
4. Fit the B − r relationship

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.vizier import Vizier

from dl import authClient as ac, queryClient as qc

## 2. Define 50 random sky regions

Criteria:
- Galactic latitude |b| > 20° (avoid galactic plane)
- −30° < Dec < 85° (main Legacy Survey DR10 footprint)
- Search radius: 0.5°

In [ ]:
SEARCH_RADIUS = 0.5  # degrees

np.random.seed(42)
regions = []
while len(regions) < 50:
    ra  = np.random.uniform(0, 360)
    dec = np.random.uniform(-30, 85)
    coord = SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame='icrs')
    if abs(coord.galactic.b.deg) > 20:
        regions.append({'ra': round(ra, 4), 'dec': round(dec, 4)})

regions_df = pd.DataFrame(regions)
print(f'{len(regions_df)} regions generated')
regions_df.head()

## 3. GLADE 2.3 query (B band)

In [ ]:
vizier = Vizier(columns=['RAJ2000', 'DEJ2000', 'Bmag', 'z', 'HyperLEDA'],
                row_limit=-1)

glade_frames = []

for i, row in regions_df.iterrows():
    coord = SkyCoord(ra=row.ra*u.deg, dec=row.dec*u.deg, frame='icrs')
    result = vizier.query_region(coord, radius=SEARCH_RADIUS*u.deg,
                                 catalog='VII/281/glade2')
    n = len(result[0]) if result else 0
    if n > 0:
        df = result[0].to_pandas()
        df['region_id'] = i
        glade_frames.append(df)
    print(f'  Region {i:2d} ({row.ra:.2f}, {row.dec:.2f}): {n} GLADE objects')

glade_all = pd.concat(glade_frames, ignore_index=True)
glade_all = glade_all.rename(columns={'RAJ2000': 'ra', 'DEJ2000': 'dec',
                                       'Bmag': 'B_mag'})
glade_all = glade_all.dropna(subset=['B_mag'])
print(f'\nTotal GLADE with Bmag: {len(glade_all)}')
glade_all[['ra', 'dec', 'B_mag', 'z']].head()

## 4. Legacy Survey DR10 query + simultaneous cross-match

For each region, we build a WHERE clause using the exact positions of the GLADE objects
found (3 arcsec radius each), avoiding downloading the entire LS region.

In [ ]:
MATCH_RAD_DEG = 3 / 3600.0  # 3 arcsec in degrees

matched_frames = []

for region_id, grp in glade_all.groupby('region_id'):
    if len(grp) == 0:
        continue

    # One q3c condition per GLADE object in this region
    conditions = " OR ".join(
        f"q3c_radial_query(ra, dec, {r.ra}, {r.dec}, {MATCH_RAD_DEG})"
        for _, r in grp.iterrows()
    )
    sql = f"""
        SELECT ra, dec, flux_r, type
        FROM   ls_dr10.tractor
        WHERE  ({conditions})
          AND  flux_r > 0
          AND  type != 'PSF'
          AND  maskbits = 0
    """
    try:
        ls_region = qc.query(sql=sql, fmt='pandas')
    except Exception as e:
        print(f'  Region {region_id:2d}: LS ERROR — {e}')
        continue

    if ls_region.empty:
        print(f'  Region {region_id:2d}: 0 LS matches')
        continue

    # Positional cross-match in Python (already small sets)
    glade_coords = SkyCoord(ra=grp['ra'].values*u.deg, dec=grp['dec'].values*u.deg)
    ls_coords    = SkyCoord(ra=ls_region['ra'].values*u.deg, dec=ls_region['dec'].values*u.deg)
    idx, sep2d, _ = glade_coords.match_to_catalog_sky(ls_coords)
    mask = sep2d < 3*u.arcsec

    if mask.sum() == 0:
        print(f'  Region {region_id:2d}: 0 matches')
        continue

    glade_matched = grp.iloc[mask].copy().reset_index(drop=True)
    ls_matched    = ls_region.iloc[idx[mask]].reset_index(drop=True)

    frame = pd.DataFrame({
        'ra':         glade_matched['ra'],
        'dec':        glade_matched['dec'],
        'B_mag':      glade_matched['B_mag'],
        'z':          glade_matched['z'],
        'r_mag':      22.5 - 2.5 * np.log10(ls_matched['flux_r'].values),
        'sep_arcsec': sep2d[mask].arcsec,
        'region_id':  region_id,
    })
    matched_frames.append(frame)
    print(f'  Region {region_id:2d}: {mask.sum()} matches')

matched = pd.concat(matched_frames, ignore_index=True)
matched = matched[(matched['r_mag'] > 10) & (matched['r_mag'] < 22)]
print(f'\nTotal matches: {len(matched)}')
matched.describe()

## 6. B − r relationship: linear fit and visualization

In [ ]:
# Remove obvious outliers (B-r difference outside 5σ)
color = matched['B_mag'] - matched['r_mag']
sigma_clip = (np.abs(color - color.median()) < 5 * color.std())
clean = matched[sigma_clip].copy()
print(f'Objects after sigma-clip: {len(clean)}')

# Linear regression r → B
slope, intercept, r_value, p_value, std_err = stats.linregress(clean['r_mag'], clean['B_mag'])
print(f'\nFit: B = {slope:.4f} * r + {intercept:.4f}')
print(f'R² = {r_value**2:.4f},  p = {p_value:.2e}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left panel: B vs r coloured by redshift ---
ax = axes[0]
sc = ax.scatter(clean['r_mag'], clean['B_mag'],
                c=clean['z'], cmap='plasma', s=20, alpha=0.7, vmin=0, vmax=0.15)
x_fit = np.linspace(clean['r_mag'].min(), clean['r_mag'].max(), 200)
ax.plot(x_fit, slope*x_fit + intercept, 'r-', lw=2,
        label=f'B = {slope:.3f}·r + {intercept:.2f}\n$R^2$ = {r_value**2:.3f}')
ax.set_xlabel('r (Legacy Survey DR10) [AB mag]', fontsize=12)
ax.set_ylabel('B (GLADE) [mag]', fontsize=12)
ax.set_title('B band vs r band — 50 random regions', fontsize=12)
ax.legend(fontsize=10)
plt.colorbar(sc, ax=ax, label='Redshift z')

# --- Right panel: B − r colour histogram ---
ax2 = axes[1]
bm_r = clean['B_mag'] - clean['r_mag']
ax2.hist(bm_r, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
ax2.axvline(bm_r.mean(), color='red', lw=2, label=f'Mean = {bm_r.mean():.2f}')
ax2.axvline(bm_r.median(), color='orange', lw=2, linestyle='--',
            label=f'Median = {bm_r.median():.2f}')
ax2.set_xlabel('B − r [mag]', fontsize=12)
ax2.set_ylabel('N galaxies', fontsize=12)
ax2.set_title('B − r colour distribution', fontsize=12)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.savefig('BvsR_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: BvsR_comparison.png')

## 7. Summary statistics

In [ ]:
print('=== Summary ===')
print(f'Regions queried          : 50')
print(f'GLADE objects (with Bmag): {len(glade_all)}')
print(f'Matches (sep < 3\'\')     : {len(matched)}')
print(f'After sigma-clip          : {len(clean)}')
print()
print(f'B − r colour')
print(f'  Mean     = {bm_r.mean():.3f} mag')
print(f'  Median   = {bm_r.median():.3f} mag')
print(f'  σ        = {bm_r.std():.3f} mag')
print()
print(f'Linear fit B = a·r + b')
print(f'  a  = {slope:.4f} ± {std_err:.4f}')
print(f'  b  = {intercept:.4f}')
print(f'  R² = {r_value**2:.4f}')